In [ ]:
pip install -U langchain

In [ ]:
pip install -U langchain-google-genai

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("gemini_api_key")
)

class SoftwareAgencyState(TypedDict):
    messages: Annotated[list, operator.add]

    user_requirement: str

    project_type: str

    requirement_analysis: str

    architecture_plan: str

    backend_plan: str

    database_plan: str

    qa_plan: str

    review_report: str

    final_report: str

In [ ]:
def requirement_analyzer(state):

    prompt = f"""
    Analyze software requirement:

    {state['user_requirement']}

    Extract:
    - Project Name
    - Features
    - Users
    - Complexity
    """

    response = llm.invoke(prompt)

    return {
        "requirement_analysis": response.content
    }

In [ ]:
def project_classifier(state):

    requirement = state["user_requirement"].lower()

    if any(word in requirement for word in
           ["ai","machine learning","llm","rag"]):

        project_type = "AI"

    elif any(word in requirement for word in
             ["website","portfolio","ecommerce"]):

        project_type = "WEB"

    else:
        project_type = "ENTERPRISE"

    return {
        "project_type": project_type
    }

In [ ]:
def ai_architect(state):

    prompt = f"""
    Design architecture for AI application.

    Requirement:
    {state['user_requirement']}

    Generate:
    - Architecture
    - AI Components
    - Models
    - Tech Stack
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [ ]:
def frontend_architect(state):

    prompt = f"""
    Design frontend architecture.

    Requirement:
    {state['user_requirement']}

    Generate:
    - UI Structure
    - Pages
    - Tech Stack
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [ ]:
def enterprise_architect(state):

    prompt = f"""
    Design enterprise architecture.

    Requirement:
    {state['user_requirement']}

    Generate:
    - Modules
    - Architecture
    - Scalability
    """

    response = llm.invoke(prompt)

    return {
        "architecture_plan": response.content
    }

In [ ]:
def backend_agent(state):

    prompt = f"""
    Architecture:

    {state['architecture_plan']}

    Generate:
    - APIs
    - Authentication
    - Folder Structure
    """

    response = llm.invoke(prompt)

    return {
        "backend_plan": response.content
    }

In [ ]:
def database_agent(state):

    prompt = f"""
    Requirement:
    {state['user_requirement']}

    Generate:
    - Tables
    - Relationships
    - Indexes
    """

    response = llm.invoke(prompt)

    return {
        "database_plan": response.content
    }

In [ ]:
def qa_agent(state):

    prompt = f"""
    Requirement:
    {state['user_requirement']}

    Generate:
    - Test Cases
    - Edge Cases
    """

    response = llm.invoke(prompt)

    return {
        "qa_plan": response.content
    }

In [ ]:
def reviewer_agent(state):

    prompt = f"""
    Review:

    Architecture:
    {state['architecture_plan']}

    Backend:
    {state['backend_plan']}

    Database:
    {state['database_plan']}

    QA:
    {state['qa_plan']}

    Return:

    APPROVED

    or

    NEEDS_REVISION

    with reason.
    """

    response = llm.invoke(prompt)

    return {
        "review_report": response.content
    }

In [ ]:
def final_report_agent(state):

    report = f"""
    SOFTWARE DEVELOPMENT PLAN

    Project Type:
    {state['project_type']}

    Requirement Analysis:
    {state['requirement_analysis']}

    Architecture:
    {state['architecture_plan']}

    Backend:
    {state['backend_plan']}

    Database:
    {state['database_plan']}

    QA:
    {state['qa_plan']}

    Review:
    {state['review_report']}
    """

    return {
        "final_report": report
    }

In [ ]:
def project_router(state):

    project_type = state["project_type"]

    if project_type == "AI":
        return "ai"

    elif project_type == "WEB":
        return "web"

    else:
        return "enterprise"

In [ ]:
workflow = StateGraph(SoftwareAgencyState)

In [ ]:
workflow.add_node(
    "Requirement Analyzer",
    requirement_analyzer
)

workflow.add_node(
    "Project Classifier",
    project_classifier
)

workflow.add_node(
    "AI Architect",
    ai_architect
)

workflow.add_node(
    "Frontend Architect",
    frontend_architect
)

workflow.add_node(
    "Enterprise Architect",
    enterprise_architect
)

workflow.add_node(
    "Backend Agent",
    backend_agent
)

workflow.add_node(
    "Database Agent",
    database_agent
)

workflow.add_node(
    "QA Agent",
    qa_agent
)

workflow.add_node(
    "Reviewer Agent",
    reviewer_agent
)

workflow.add_node(
    "Final Report",
    final_report_agent
)

In [ ]:
workflow.set_entry_point(
    "Requirement Analyzer"
)

In [ ]:
workflow.add_edge(
    "Requirement Analyzer",
    "Project Classifier"
)

In [ ]:
workflow.add_conditional_edges(
    "Project Classifier",
    project_router,
    {
        "ai": "AI Architect",

        "web": "Frontend Architect",

        "enterprise": "Enterprise Architect"
    }
)

In [ ]:
workflow.add_edge(
    "AI Architect",
    "Backend Agent"
)

workflow.add_edge(
    "Frontend Architect",
    "Backend Agent"
)

workflow.add_edge(
    "Enterprise Architect",
    "Backend Agent"
)

In [ ]:
workflow.add_edge(
    "Backend Agent",
    "Database Agent"
)

workflow.add_edge(
    "Database Agent",
    "QA Agent"
)

workflow.add_edge(
    "QA Agent",
    "Reviewer Agent"
)

workflow.add_edge(
    "Reviewer Agent",
    "Final Report"
)

workflow.add_edge(
    "Final Report",
    END
)

In [ ]:
app = workflow.compile()
result = app.invoke({
    "user_requirement":
    "Build an AI Interview Assistant using LLM and RAG"
})